# Phase 2 Notebook: Candidate Generation Fernsehserien.de
This notebook orchestrates the full fernsehserien.de candidate-generation pipeline.
It supports discovery, extraction, normalization, and replay across all eligible programs in setup input, with configurable runtime budgets.

## How This Implementation Works (Quick Guide)

This notebook runs one deterministic fernsehserien workflow per execution:
1. Resolve known episode leaf URLs.
2. Fetch missing leaf pages (cache-first), then continue resolution.
3. Fetch additional episode URLs from index traversal as needed, then continue leaf fetching.
4. Normalize discovered data.
5. Build and verify projections.

Network behavior is controlled only by `MAX_NETWORK_CALLS`:
- `0`: cache-only execution (no new network calls).
- `>0`: bounded network execution.
- `<0`: unlimited network budget.

### Where Sequence Numbers Are Persisted
The canonical sequence numbers are stored in the append-only event log:
- `data/20_candidate_generation/fernsehserien_de/chunks/chunk_000001.jsonl`
- each event contains `sequence_num`.

### Per-Handler Progress Table
The projection handler persists progress to:
- `data/20_candidate_generation/fernsehserien_de/eventhandler.csv`

Schema:
- `handler_name,last_processed_sequence,artifact_path,updated_at`

Current handler behavior:
- reads `last_processed_sequence` for each projection handler,
- processes only events with higher sequence numbers,
- writes updated projections, then updates `eventhandler.csv`.

This gives incremental resume behavior while keeping the event log as source of truth.

## 1) Project Setup
Resolve repository root and make `speakermining/src` importable for full-pipeline orchestration.

In [ ]:
# Optional projection reset only: cache pages and events must never be deleted.
import shutil
from pathlib import Path


RESET_PROJECTIONS = False


def find_repo_root_for_cleanup(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "data").exists() and (cur / "speakermining" / "src").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Repository root not found for cleanup.")


cleanup_root = find_repo_root_for_cleanup(Path.cwd())
runtime_root = cleanup_root / "data" / "20_candidate_generation" / "fernsehserien_de"
projection_root = runtime_root / "projections"

if RESET_PROJECTIONS:
    if projection_root.exists():
        shutil.rmtree(projection_root)
        print(f"Deleted derived projections only: {projection_root}")
    else:
        print("No projections directory to delete")
else:
    print("Projection reset skipped (RESET_PROJECTIONS=False); cache/events are preserved")

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "data").exists() and (cur / "speakermining" / "src").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Repository root not found.")


ROOT = find_repo_root(Path.cwd())
SRC = ROOT / "speakermining" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# ROOT

## 2) Configure Runtime Parameters
Set only the core runtime parameters for production execution.
- `MAX_NETWORK_CALLS` controls cache-only vs bounded/unlimited network behavior.
- `QUERY_DELAY_SECONDS` controls request pacing.
- `USER_AGENT` identifies this pipeline to the service.

In [ ]:
from process.notebook_event_log import get_or_create_notebook_logger
from process.candidate_generation.fernsehserien_de import FernsehserienRunConfig
import pandas as pd

NOTEBOOK_ID = "notebook_22_candidate_generation_fernsehserien_de"
logger = get_or_create_notebook_logger(ROOT, NOTEBOOK_ID)

programs_path = ROOT / "data" / "00_setup" / "broadcasting_programs.csv"
programs_df = pd.read_csv(programs_path)
fernsehserien_ids = programs_df["fernsehserien_de_id"].fillna("").astype(str).str.strip()
eligible_programs_df = programs_df[
    fernsehserien_ids.ne("")
    & (~fernsehserien_ids.str.upper().isin({"NONE", "NULL", "NAN"}))
] .copy()

# Core runtime configuration
MAX_NETWORK_CALLS = 100  # 0=cache-only, >0=bounded network, <0=unlimited
QUERY_DELAY_SECONDS = 1.0
USER_AGENT = "speaker-mining/0.1 (fernsehserien-stage2-pipeline) python/3"

print(f"Eligible programs: {len(eligible_programs_df)}")
print(f"MAX_NETWORK_CALLS: {MAX_NETWORK_CALLS}")
print(f"QUERY_DELAY_SECONDS: {QUERY_DELAY_SECONDS}")
print(f"USER_AGENT: {USER_AGENT}")

## 3) Execute Workflow
Run the pipeline once using the configured `MAX_NETWORK_CALLS` budget.

In [ ]:
from process.candidate_generation.fernsehserien_de import run_fernsehserien_pipeline

cfg = FernsehserienRunConfig(
    repo_root=ROOT,
    query_delay_seconds=QUERY_DELAY_SECONDS,
    max_network_calls=int(MAX_NETWORK_CALLS),
    max_programs=None,
    user_agent=USER_AGENT,
)

final_result = run_fernsehserien_pipeline(config=cfg, notebook_logger=logger)

print(
    f"programs_processed={final_result['programs_processed']} "
    f"network_calls_used={final_result['network_calls_used']} "
    f"max_network_calls={final_result['max_network_calls']} "
    f"normalized_events_emitted={final_result.get('normalized_events_emitted', 0)}"
)

# Emit exactly one run_finished event per logger run_id, even if this cell is re-executed.
if "_NOTEBOOK22_FINISHED_RUN_IDS" not in globals():
    _NOTEBOOK22_FINISHED_RUN_IDS = set()

if logger.run_id not in _NOTEBOOK22_FINISHED_RUN_IDS:
    logger.append_event(
        event_type="run_finished",
        phase="run_lifecycle",
        message="notebook run finished",
        budget={
            "max_queries_per_run": int(final_result["max_network_calls"]),
            "queries_used_after": int(final_result["network_calls_used"]),
        },
        result={
            "programs_processed": int(final_result["programs_processed"]),
            "network_calls_used": int(final_result["network_calls_used"]),
            "max_network_calls": int(final_result["max_network_calls"]),
            "normalized_events_emitted": int(final_result.get("normalized_events_emitted", 0)),
            "status": "success",
        },
    )
    _NOTEBOOK22_FINISHED_RUN_IDS.add(logger.run_id)
    print("Lifecycle event emitted: run_finished")
else:
    print("Lifecycle event already emitted for this run_id; skipping duplicate run_finished")

## 4) Verify Run Behavior
Validate that runtime behavior matches the configured `MAX_NETWORK_CALLS` policy.

In [ ]:
verification = {
    "eligible_programs": int(len(eligible_programs_df)),
    "programs_processed": int(final_result["programs_processed"]),
    "network_calls_used": int(final_result["network_calls_used"]),
    "max_network_calls": int(final_result["max_network_calls"]),
}

if int(MAX_NETWORK_CALLS) == 0:
    assert int(final_result["network_calls_used"]) == 0, (
        f"Cache-only run used network unexpectedly: {final_result['network_calls_used']}"
    )
elif int(MAX_NETWORK_CALLS) > 0:
    assert int(final_result["network_calls_used"]) <= int(MAX_NETWORK_CALLS), (
        "Network calls used exceeded configured budget"
    )

print("Verification:", verification)
verification

## 5) Inspect Produced Projections
Load discovered and normalized projection artifacts produced by the final pipeline pass.

In [ ]:
import json
import pandas as pd
from IPython.display import display

runtime_root = ROOT / "data" / "20_candidate_generation" / "fernsehserien_de"
projection_root = runtime_root / "projections"

program_pages_df = pd.read_csv(projection_root / "program_pages.csv")
episode_index_df = pd.read_csv(projection_root / "episode_index_pages.csv")
episode_urls_df = pd.read_csv(projection_root / "episode_urls.csv")
metadata_discovered_df = pd.read_csv(projection_root / "episode_metadata_discovered.csv")
guests_discovered_df = pd.read_csv(projection_root / "episode_guests_discovered.csv")
broadcasts_discovered_df = pd.read_csv(projection_root / "episode_broadcasts_discovered.csv")
metadata_normalized_df = pd.read_csv(projection_root / "episode_metadata_normalized.csv")
guests_normalized_df = pd.read_csv(projection_root / "episode_guests_normalized.csv")
broadcasts_normalized_df = pd.read_csv(projection_root / "episode_broadcasts_normalized.csv")
summary = json.loads((projection_root / "summary.json").read_text(encoding="utf-8"))

print("Projection summary:", summary)
print("Configured max_network_calls:", int(MAX_NETWORK_CALLS))
print("Run network usage:", final_result["network_calls_used"], "/", final_result["max_network_calls"])
print("Programs processed:", final_result["programs_processed"])

print("program_pages.csv")
display(program_pages_df)
print("episode_index_pages.csv")
display(episode_index_df)
print("episode_urls.csv")
display(episode_urls_df)
print("episode_metadata_discovered.csv")
display(metadata_discovered_df)
print("episode_guests_discovered.csv")
display(guests_discovered_df)
print("episode_broadcasts_discovered.csv")
display(broadcasts_discovered_df)
print("episode_metadata_normalized.csv")
display(metadata_normalized_df)
print("episode_guests_normalized.csv")
display(guests_normalized_df)
print("episode_broadcasts_normalized.csv")
display(broadcasts_normalized_df)